# 1. Onboarding: identities, credentials and peer tokens

Before two connectors can speak the Dataspace Protocol (DSP 2025-1) they must know who the other
one is and hold a token to present on every DSP call. This notebook onboards the three
participants of the dev dataspace:

| Participant | Default URL | Role |
|---|---|---|
| Authority | `http://127.0.0.1:1500` | Issues the `DataSpaceParticipant` verifiable credential |
| Consumer | `http://127.0.0.1:1100` | Discovers the catalog, negotiates and receives data |
| Provider | `http://127.0.0.1:1200` | Publishes datasets, agrees contracts and serves data |

Steps:
1. Configuration and the helpers every other notebook uses.
2. Link the wallet of each participant (keys and DID document).
3. Read each participant's DID from `/.well-known/did.json`.
4. Consumer and Provider obtain a credential from the Authority.
5. Consumer and Provider connect to each other and exchange peer tokens (GNAP).
6. Resolve the Consumer's own DSP base URL, used as `callbackAddress` in negotiations and transfers.

**Runs on its own.** It has no prerequisites; every other notebook runs it first through a guarded
`%run`, so it executes once per kernel. Steps 2, 4 and 5 are idempotent: rerunning them reuses
what the agents already hold.

## 1. Configuration and helpers

The URLs can be overridden with environment variables. The `INTERNAL_*` variants are the addresses
the agents use to reach each other (they differ from the public ones when the agents run inside
Docker).

`show(response)` prints an HTTP answer as JSON and fails the cell on a non-2xx status, so a broken
step stops a chained `%run` instead of feeding empty ids to the next one. Pass `ok=False` when a
failure is the expected result.

In [ ]:
import json
import os
import time

import requests

data_space_authority = os.environ.get("AUTHORITY_URL", "http://127.0.0.1:1500")
data_space_consumer = os.environ.get("CONSUMER_URL", "http://127.0.0.1:1100")
data_space_provider = os.environ.get("PROVIDER_URL", "http://127.0.0.1:1200")

internal_authority_url = os.environ.get("INTERNAL_AUTHORITY_URL", data_space_authority)
internal_consumer_url = os.environ.get("INTERNAL_CONSUMER_URL", data_space_consumer)
internal_provider_url = os.environ.get("INTERNAL_PROVIDER_URL", data_space_provider)

json_header = {"Content-Type": "application/json"}

# JSON-LD context of every Eunomia RPC body (negotiation and transfer)
eunomia_context = "https://w3id.org/eunomia/v1/context.jsonld"


def show(response, ok=True):
    """Prints an HTTP answer as JSON; with ok, raises unless the status is 2xx."""
    try:
        body = response.json()
    except ValueError:
        body = response.text
    print(f"HTTP {response.status_code} {response.request.method} {response.url}")
    print(body if isinstance(body, str) else json.dumps(body, indent=2))
    if ok and not response.ok:
        raise RuntimeError(f"{response.request.method} {response.url} answered {response.status_code}")
    return body


print(f"Authority URL: {data_space_authority}")
print(f"Consumer URL:  {data_space_consumer}")
print(f"Provider URL:  {data_space_provider}")

## 2. Link wallets

Each agent links its wallet, which creates its keys and publishes its DID document. Linking an
already linked wallet only answers a warning.

In [ ]:
for name, agent_url in [("Authority", data_space_authority), ("Consumer", data_space_consumer), ("Provider", data_space_provider)]:
    res = requests.post(f"{agent_url}/api/v1/wallet/link", headers=json_header, json={})
    if res.status_code in (200, 204):
        print(f"Wallet linked for {name} ({agent_url})")
    else:
        print(f"Wallet of {name} not linked now ({res.status_code}): {res.text}")

## 3. Participant DIDs

A participant is identified in the dataspace by its decentralized identifier (DID). The DSP calls
below name the peer by this DID in `associatedAgentPeer`.

In [ ]:
data_space_authority_participant_id = requests.get(f"{data_space_authority}/.well-known/did.json").json()["id"]
data_space_consumer_participant_id = requests.get(f"{data_space_consumer}/.well-known/did.json").json()["id"]
data_space_provider_participant_id = requests.get(f"{data_space_provider}/.well-known/did.json").json()["id"]

print(f"Authority DID: {data_space_authority_participant_id}\n")
print(f"Consumer DID:  {data_space_consumer_participant_id}\n")
print(f"Provider DID:  {data_space_provider_participant_id}")

## 4. Credentials from the Authority

Consumer and Provider ask the Authority for a `DataSpaceParticipant_jwt_vc_json` credential
through its gate (`/api/v1/gate/access`). With `auto: true` the Authority issues it at once. A
participant that already holds a finalized credential skips the request.

In [ ]:
vc_type = "DataSpaceParticipant_jwt_vc_json"


def has_credential(agent_url):
    """Whether the agent holds a finalized credential of vc_type from the Authority."""
    items = requests.get(f"{agent_url}/api/v1/vc-request/all", headers=json_header).json().get("items", [])
    return any(
        req.get("participant_id") == data_space_authority_participant_id
        and req.get("status") == "Finalized"
        and vc_type in (req.get("vc_type_config") or [])
        for req in items
    )


def request_credential(agent_name, agent_url):
    if has_credential(agent_url):
        print(f"{agent_name} already holds {vc_type} from the Authority.")
        return
    beg_body = {
        "url": f"{internal_authority_url}/api/v1/gate/access",
        "id": data_space_authority_participant_id,
        "nick": "authority",
        "vc_type": vc_type,
        "method": "cert",
        "auto": True,
    }
    res = requests.post(f"{agent_url}/api/v1/vc-request/beg", json=beg_body, headers=json_header)
    if res.status_code not in (200, 201, 204):
        raise RuntimeError(f"{agent_name} could not request a credential: {res.status_code} - {res.text}")
    if not has_credential(agent_url):
        raise RuntimeError(f"Credential issuance for {agent_name} did not finalize.")
    print(f"{agent_name} obtained {vc_type} from the Authority.")


request_credential("Consumer", data_space_consumer)
request_credential("Provider", data_space_provider)

## 5. Mutual peer connection (Consumer <-> Provider)

Each side connects to the other's gate presenting its credential, and receives an access token
in return. Both directions are needed: the Consumer calls the Provider (catalog, negotiation,
transfer requests) and the Provider calls the Consumer back (offers, agreements, transfer starts).

In [ ]:
def approved_token(source_url, target_did):
    """The token source_url holds for target_did, or None if not connected yet."""
    items = requests.get(f"{source_url}/api/v1/peer-connection/request/all", headers=json_header).json().get("items", [])
    for req in reversed(items):
        if req.get("participant_id") == target_did and req.get("status") == "Approved" and req.get("final_token"):
            return req["final_token"]
        if req.get("participant_id") == target_did and req.get("status") == "Rejected":
            raise RuntimeError(f"Connection to {target_did} was rejected.")
    return None


def connect_peer(source_name, source_url, target_name, target_gate_url, target_did):
    if approved_token(source_url, target_did):
        print(f"{source_name} is already connected to {target_name}.")
        return
    connect_body = {
        "id": target_did,
        "nick": target_name,
        "url": target_gate_url,
        "actions": ["talk"],
        "visibility": "Public",
        "auto": True,
    }
    res = requests.post(f"{source_url}/api/v1/peer-connection/connect", json=connect_body, headers=json_header)
    if res.status_code not in (200, 201, 204):
        raise RuntimeError(f"{source_name} could not connect to {target_name}: {res.status_code} - {res.text}")
    for _ in range(30):
        if approved_token(source_url, target_did):
            print(f"{source_name} connected to {target_name}.")
            return
        time.sleep(1)
    raise TimeoutError(f"Timeout waiting for {source_name} to connect to {target_name}")


connect_peer("Consumer", data_space_consumer, "provider", f"{internal_provider_url}/api/v1/gate/access", data_space_provider_participant_id)
connect_peer("Provider", data_space_provider, "consumer", f"{internal_consumer_url}/api/v1/gate/access", data_space_consumer_participant_id)

## 6. Peer tokens and the Consumer's DSP address

The agents attach these tokens to DSP calls by themselves; they are printed here only to show
what onboarding produced:
- `data_space_provider_participant_token`: held by the Consumer, presented to the Provider.
- `data_space_consumer_participant_token`: held by the Provider, presented to the Consumer.

The Consumer's DSP base URL comes from its version discovery endpoint
(`/.well-known/dspace-version/2025-1`). It is the `callbackAddress` the Provider answers to.

In [ ]:
data_space_provider_participant_token = approved_token(data_space_consumer, data_space_provider_participant_id)
data_space_consumer_participant_token = approved_token(data_space_provider, data_space_consumer_participant_id)

version = requests.get(f"{data_space_consumer}/.well-known/dspace-version/2025-1", headers=json_header).json()
data_space_consumer_dsp_endpoint = data_space_consumer + version["path"]

# Marker read by the guards of the other notebooks
onboarded = True

print(f"Provider token (held by Consumer): {data_space_provider_participant_token}\n")
print(f"Consumer token (held by Provider): {data_space_consumer_participant_token}\n")
print(f"Consumer DSP address:              {data_space_consumer_dsp_endpoint}")